# Faithfulness under zero, mean and blur fills (re-run of the perscan faithfulness stage)

Same frozen SFCN, same cached features, same seeded age head as the perscan run. Attribution maps are recomputed
(not saved earlier). Adds mean-intensity and Gaussian-blur fills; zero fill is re-run as a reproducibility check.
Pre-registered in `docs/prenatal_xai_plan.md` section 10 (A7, A8).

# Validating explanations on a pediatric SFCN (public Calgary preschool controls)

**One line.** Das et al. (2024) flagged their explainability as *preliminary* — a single coarse Grad-CAM map, with only one of four named regions tested statistically. This notebook builds the rigorous version: a pre-registered battery that tests which explanation methods are **faithful**, **stable across methods**, and **anatomically valid** on *their exact model class*.

**Same model, proxy task.** We use the *same* network they used — the pretrained SFCN (Peng et al., 2021), frozen, six convolutional blocks kept as a feature extractor. The only differences are the small head on top and the label it predicts:

| | Das et al. (2024) | This study |
|---|---|---|
| Backbone | Pretrained SFCN, frozen | **Same** pretrained SFCN, frozen |
| Head | small dense head → PAE | small dense head → **age group** (older vs younger, median split) |
| Data | private PAE + public controls | **public controls only (96 children)** |

Swapping the head's task does **not** invalidate the XAI. Explanation *validity* — faithfulness, the randomized-weight sanity check, cross-method agreement, resolution — is a property of the **method and the architecture**, not of the task. Testing it on a controllable public proxy is standard practice (Adebayo et al. 2018; Siegel et al. 2025).

**What this does and does not show.**

| Claim | This notebook |
|---|---|
| Which XAI methods are trustworthy on the SFCN | **Yes — the deliverable** |
| Grad-CAM over-resolves / methods disagree / which fail sanity | **Yes — the headline** |
| Anything about alcohol exposure, FASD, PAE | **No — needs the private cohort** |
| Reproducing the paper's 85–88% accuracy | **No — that was the PAE task** |

The second column is the **next stage** and the reason to collaborate — not a gap in this work.


## How this notebook is run

**Preprocessing is already done.** All 96 children have been skull-stripped, N4-corrected, affine-registered to MNI152 1 mm and cropped to 160×192×160 on Kaggle (two batches), and all 96 passed visual QC. The preprocessed volumes (`preproc/*.npy`), the cached frozen feature vectors (`features/*.npy`), the pretrained SFCN weights, the labels (`manifest.csv`) and the Harvard–Oxford atlas are supplied as an **attached Kaggle dataset**. This notebook therefore **does not re-run ANTs** and **does not re-extract features** — it loads them and spends GPU only on attributions and the masking test.

**First-run checks** (marked `# FIRST-RUN CHECK`): (1) the attached dataset is found and resolves all 96 preprocessed volumes + labels; (2) the SFCN `feature_extractor.*` keys all load; (3) the feature-map geometry is 64×5×6×5; (4) the atlas lands on the same 160×192×160 crop. Each stage caches to `/kaggle/working`, so a re-run resumes.

**Modes.** `full` (default) runs all 96 children with pre-registered settings — the deliverable. `smoke` shrinks every knob to a handful of scans to prove the plumbing in minutes; its numbers are **not** interpretable.


In [ ]:
# === Configuration (pre-registered values; changing one goes in the decision log) ===
from pathlib import Path

CFG = dict(
    MODE="full",                 # "full" (all 96 children) | "smoke" (plumbing check)
    USE_PRECOMPUTED=True,         # consume attached preprocessed volumes + cached features
    EXPLAIN_SET="all",            # "all" (every child) | "test_fold" (one grouped fold)
    ONE_SCAN_PER_CHILD=True,
    SEED=0,
    EXPECTED_N=96,            # hard guard: full run MUST see exactly 96 children

    # --- data / labels ---
    PROXY_LABEL="age",            # proxy = age dichotomized at the cohort median (older=1)
    SFCN_INPUT=(160, 192, 160),   # 1 mm MNI crop; yields a 64 x 5 x 6 x 5 feature map

    # --- model weights (used if not found in the attached dataset) ---
    SFCN_WEIGHTS_URL=(
        "https://github.com/ha-ha-ha-han/UKBiobank_deep_pretrain/raw/master/"
        "brain_age/run_20190719_00_epoch_best_mae.p"
    ),

    # --- attribution hyperparameters (capped per plan; logged) ---
    IG_STEPS=32, SG_SAMPLES=20, SG_STDEV=0.1,
    REX_GRID=(5, 6, 5), LRP_RULE="EpsilonPlusFlat",

    # --- statistics ---
    N_BOOT=10000, N_PERM=10000, N_NULL=500, FDR_Q=0.05,
    MASK_FILLS={"mean": None, "zero": 0.0},   # zero fill checks for mask artifacts
    NULL_SCANS=4,                             # scans subsampled per random null region

    # --- CV / head ---
    N_FOLDS=5, N_REPEATS=10, N_PERM_ACC=0,    # skip ~1.3h probe-permutation; p=0.01 already in main run
    HEAD_EPOCHS=100, HEAD_LR=1e-3,
    SMOKE_MAX_REGIONS=6,
    REGION_HALF=None,            # None=all regions | 0=first half | 1=second half (split runs)
)

# SMOKE: shrink every expensive knob (plumbing check only; numbers not interpretable).
if CFG["MODE"] == "smoke":
    CFG.update(N_BOOT=200, N_PERM=200, N_NULL=15, N_FOLDS=2, N_REPEATS=2,
               N_PERM_ACC=20, HEAD_EPOCHS=15, IG_STEPS=8, SG_SAMPLES=4,
               EXPLAIN_SET="test_fold", SMOKE_N=4)
    print("=== SMOKE MODE === results are a plumbing check, not a finding.")

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
DIRS = {k: ROOT / k for k in ["raw", "preproc", "features", "explanations", "results", "qc"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("MODE =", CFG["MODE"], "| precomputed =", CFG["USE_PRECOMPUTED"], "| workspace =", ROOT)

In [ ]:
# === Dependencies (quiet; skip if already present) ===
import importlib, subprocess, sys

def _ensure(pkgs):
    for mod, pip_name in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

# Precomputed path needs NO ANTs/antspynet (no re-registration): only attribution +
# atlas libraries. nibabel stays for the local-dev fallback.
deps = [("torch", "torch"), ("captum", "captum"), ("zennit", "zennit"),
        ("nibabel", "nibabel"), ("scipy", "scipy"), ("sklearn", "scikit-learn"),
        ("nilearn", "nilearn")]
if not CFG["USE_PRECOMPUTED"]:
    deps += [("ants", "antspyx"), ("antspynet", "antspynet")]
_ensure(deps)
print("dependencies ready |", "precomputed (no ANTs)" if CFG["USE_PRECOMPUTED"] else "heavy (ANTs)")

In [ ]:
# === Imports, determinism, device ===
import json, os, time, copy, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

## Pre-registration — the locked rules (shown, not hidden)

These were fixed in `prenatal_xai_plan.md` **before** any attribution was computed. The cell below prints the frozen configuration so a reader can see exactly what was committed.

**Hypotheses.** H1 Grad-CAM (M1) is coarser than input-resolution methods (M4/M5). · H2 only large regions survive masking + FDR; small named structures (pons, corpus callosum) do not. · H3 M4 and M5 agree with each other more than with M1. · H4 for at least one method, randomized-weight maps differ from trained maps (sanity). · H5 a block-6 probe beats a block-2 probe. · H6 age-group accuracy beats chance under a child-level label-permutation test.

**Method set (locked):** M1 Grad-CAM final conv (paper's method) · M2 Grad-CAM early block · M3 Saliency · M4 Integrated Gradients · M5 SmoothGrad · M6 VarGrad · M7 LRP (zennit) · M8 Guided Backprop (negative control). **M9 3D ReX is exploratory**, added after pre-registration and logged as such.


In [ ]:
# === Show the frozen pre-registered settings ===
_show = ["MODE","EXPLAIN_SET","ONE_SCAN_PER_CHILD","SEED","PROXY_LABEL","SFCN_INPUT",
         "IG_STEPS","SG_SAMPLES","SG_STDEV","REX_GRID","LRP_RULE",
         "N_BOOT","N_PERM","N_NULL","FDR_Q","N_FOLDS","N_REPEATS","N_PERM_ACC",
         "HEAD_EPOCHS","HEAD_LR"]
print("Frozen configuration")
print("-" * 40)
for k in _show:
    print(f"  {k:22s} = {CFG[k]}")
print(f"  {'MASK_FILLS':22s} = {list(CFG['MASK_FILLS'])}")

## 1. Data — public Calgary Preschool T1 (controls only), 96 children

Labels (`child_id, scan_id, age`) come from the attached manifest; the proxy label is age dichotomized at the cohort median (older=1, younger=0). One scan per child (no leakage). The attached dataset supplies the preprocessed volumes directly, so no download happens on the full path.


In [ ]:
# === Locate the attached dataset: manifest + preprocessed volumes (+ cached features) ===
def _find_under_input(pattern):
    base = Path("/kaggle/input")
    return sorted(base.rglob(pattern)) if base.exists() else []

def load_manifest_precomputed():
    """FIRST-RUN CHECK: find manifest.csv with labels + a preproc/<scan_id>.npy for each row."""
    # Prefer a manifest that sits next to a preproc/ folder.
    cand = _find_under_input("manifest.csv")
    if not cand:
        raise FileNotFoundError("No manifest.csv under /kaggle/input — attach the preprocessed dataset.")
    man_path = cand[0]
    for c in cand:                                   # prefer one with a sibling preproc/
        if (c.parent / "preproc").exists():
            man_path = c; break
    man = pd.read_csv(man_path, dtype={"child_id": str, "scan_id": str})
    base = man_path.parent

    def _resolve(scan_id, sub):
        p = base / sub / f"{scan_id}.npy"
        if p.exists():
            return str(p)
        hits = _find_under_input(f"{scan_id}.npy")
        hits = [h for h in hits if f"/{sub}/" in str(h).replace(os.sep, "/")]
        return str(hits[0]) if hits else ""

    man["preproc_npy"] = man["scan_id"].apply(lambda s: _resolve(s, "preproc"))
    man["feature_npy"] = man["scan_id"].apply(lambda s: _resolve(s, "features"))
    print("manifest:", man_path)
    print("rows:", len(man),
          "| preproc found:", (man["preproc_npy"] != "").sum(),
          "| features found:", (man["feature_npy"] != "").sum())
    missing = man[man["preproc_npy"] == ""]["scan_id"].tolist()
    if missing:
        raise FileNotFoundError(f"preprocessed volume missing for: {missing[:10]} ...")
    return man

if CFG["USE_PRECOMPUTED"]:
    manifest = load_manifest_precomputed()
    # Stage volumes into DIRS["preproc"] under <scan_id>.npy so the rest of the
    # notebook reads them by scan_id (works whether or not features are cached).
    for _, row in manifest.iterrows():
        dst = DIRS["preproc"] / f"{row['scan_id']}.npy"
        if not dst.exists():
            try:
                os.symlink(row["preproc_npy"], dst)
            except OSError:
                import shutil; shutil.copy(row["preproc_npy"], dst)
        if row["feature_npy"]:
            fdst = DIRS["features"] / f"{row['scan_id']}.npy"
            if not fdst.exists():
                try:
                    os.symlink(row["feature_npy"], fdst)
                except OSError:
                    import shutil; shutil.copy(row["feature_npy"], fdst)
else:
    raise RuntimeError("Set USE_PRECOMPUTED=True and attach the dataset, or add an OSF download cell.")

if CFG["MODE"] == "smoke":
    manifest = manifest.head(CFG["SMOKE_N"]).reset_index(drop=True)

# Proxy label = age dichotomized at the cohort median (older child = 1, younger = 0).
# The frozen SFCN is a brain-AGE model, so age is the signal its features encode;
# sex was near chance and is dropped as the task (decision log D11).
_age = manifest["age"].astype(float)
_median_age = _age.median()
manifest["proxy_label"] = (_age > _median_age).astype("Int64")
manifest = manifest.dropna(subset=["proxy_label"]).reset_index(drop=True)
print(f"proxy = age > median ({_median_age:.2f} y): older=1, younger=0")
print(manifest[["child_id", "scan_id", "age", "proxy_label"]].head())
# HARD GUARD against accidentally attaching the 85-scan raw 'upload' dataset.
if CFG["MODE"] == "full" and len(manifest) != CFG["EXPECTED_N"]:
    raise AssertionError(
        f"COHORT = {len(manifest)}, expected {CFG['EXPECTED_N']}. You probably attached the "
        "85-scan raw 'upload' dataset instead of the 96-scan preprocessed one. "
        "Attach the dataset built from prep_output/ + data/manifest.csv.")
print(f"COHORT CHECK OK: {len(manifest)} children (expected {CFG['EXPECTED_N']}).")
print("scans:", len(manifest), "| children:", manifest["child_id"].nunique(),
      "| age-group balance (0=younger,1=older):", np.bincount(manifest["proxy_label"].astype(int)))

## 2. Preprocessing — already complete (verification + QC montage)

No ANTs runs here. The cell verifies every volume is 160×192×160 and finite, then shows a contact sheet of the mid-sagittal slice of **all 96** preprocessed brains so a reader can confirm the registrations by eye. (The stop rule — fail if >25% of scans are bad — was applied during the original preprocessing; all 96 passed.)


In [ ]:
# === Verify preprocessed volumes + QC contact sheet of all scans ===
def load_volume(scan_id):
    vol = np.load(DIRS["preproc"] / f"{scan_id}.npy").astype(np.float32)
    return torch.from_numpy(vol)[None, None]

shapes, bad = [], []
for sid in manifest["scan_id"]:
    v = np.load(DIRS["preproc"] / f"{sid}.npy")
    shapes.append(v.shape)
    if v.shape != CFG["SFCN_INPUT"] or not np.isfinite(v).all():
        bad.append(sid)
print("all shapes == 160x192x160:", all(s == CFG["SFCN_INPUT"] for s in shapes),
      "| non-conforming/nonfinite:", bad or "none")

def montage(scan_ids, slicer, title, cmap="gray", ncol=12, per=1.1):
    n = len(scan_ids); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * per, nrow * per))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis("off")
    for ax, sid in zip(axes, scan_ids):
        ax.imshow(slicer(sid), origin="lower", cmap=cmap)
        ax.set_title(str(sid), fontsize=5)
    fig.suptitle(title, y=1.0, fontsize=11)
    fig.tight_layout(); return fig

def _mid_sag(sid):
    v = np.load(DIRS["preproc"] / f"{sid}.npy")
    return v[v.shape[0] // 2].T

fig = montage(manifest["scan_id"].tolist(), _mid_sag,
              f"Preprocessed mid-sagittal slice — all {len(manifest)} children (QC)")
fig.savefig(DIRS["qc"] / "qc_contact_sheet.png", dpi=90, bbox_inches="tight")
plt.show()

## 3. Model — the original SFCN, frozen (Peng et al., 2021)

The released brain-age checkpoint is loaded and **only the six convolutional blocks** are kept as a frozen feature extractor — exactly what Das et al. did; the classifier is discarded. The age head (§5) is separate.


In [ ]:
# === SFCN architecture + frozen feature extractor + age head ===
class SFCN(nn.Module):
    """Peng et al. (2021) SFCN; only feature_extractor is reused here."""
    def __init__(self, channel_number=(32, 64, 128, 256, 256, 64), output_dim=40, dropout=True):
        super().__init__()
        n = len(channel_number)
        self.feature_extractor = nn.Sequential()
        for i in range(n):
            ic = 1 if i == 0 else channel_number[i - 1]
            oc = channel_number[i]; mp = i < n - 1
            ks, pad = (3, 1) if mp else (1, 0)
            self.feature_extractor.add_module(f"conv_{i}", self._block(ic, oc, mp, ks, pad))
        self.classifier = nn.Sequential()
        self.classifier.add_module("average_pool", nn.AvgPool3d((5, 6, 5)))
        if dropout:
            self.classifier.add_module("dropout", nn.Dropout(0.5))
        self.classifier.add_module(f"conv_{n}", nn.Conv3d(channel_number[-1], output_dim, 1))

    @staticmethod
    def _block(ic, oc, maxpool, ks, pad):
        layers = [nn.Conv3d(ic, oc, kernel_size=ks, padding=pad), nn.BatchNorm3d(oc)]
        if maxpool: layers.append(nn.MaxPool3d(2, stride=2))
        layers.append(nn.ReLU())
        return nn.Sequential(*layers)

    def forward(self, x):
        return F.log_softmax(self.classifier(self.feature_extractor(x)), dim=1)


class AgeSFCN(nn.Module):
    """Frozen feature extractor + 512-unit head (2-logit softmax; see decision log)."""
    def __init__(self, feature_extractor, n_features=64 * 5 * 6 * 5, hidden=512, n_classes=2):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.head(self.feature_extractor(x))


def find_weights():
    for name in ("sfcn_brainage.p", "run_20190719_00_epoch_best_mae.p"):
        hits = _find_under_input(name)
        if hits:
            return hits[0]
    ckpt = DIRS["raw"] / "sfcn_brainage.p"
    if not ckpt.exists():
        print("weights not in attached dataset — downloading from GitHub ...")
        urllib.request.urlretrieve(CFG["SFCN_WEIGHTS_URL"], ckpt)
    return ckpt

def load_feature_extractor():
    ckpt = find_weights()
    model = SFCN()
    state = torch.load(ckpt, map_location="cpu")
    state = state.get("state_dict", state)
    state = {k.replace("module.", ""): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    fe_missing = [k for k in missing if k.startswith("feature_extractor.")]
    assert not fe_missing, f"feature-extractor keys unmatched: {fe_missing}"   # FIRST-RUN CHECK
    for p in model.feature_extractor.parameters():
        p.requires_grad_(False)
    model.feature_extractor.to(DEVICE).eval()
    print("feature extractor loaded from", ckpt, "| classifier keys dropped:", len(unexpected))
    return model.feature_extractor

feature_extractor = load_feature_extractor()

with torch.no_grad():
    fmap = feature_extractor(torch.zeros(1, 1, *CFG["SFCN_INPUT"], device=DEVICE))
print("feature-map shape:", tuple(fmap.shape))
assert tuple(fmap.shape[1:]) == (64, 5, 6, 5), "geometry mismatch — check SFCN_INPUT"   # FIRST-RUN CHECK

## 4. Features — use the cached frozen feature vectors

The 64×5×6×5 = 9600-dim vectors were cached during preprocessing. The cell loads them (no GPU forward pass) and only recomputes any that are missing.


In [ ]:
# === Load cached features (recompute any that are missing) ===
@torch.no_grad()
def get_features(manifest):
    feats, recomputed = {}, 0
    for _, row in manifest.iterrows():
        cache = DIRS["features"] / f"{row['scan_id']}.npy"
        if cache.exists():
            feats[row["scan_id"]] = np.load(cache).astype(np.float32); continue
        x = load_volume(row["scan_id"]).to(DEVICE)
        f = feature_extractor(x).flatten().cpu().numpy().astype(np.float32)
        np.save(cache, f); feats[row["scan_id"]] = f; recomputed += 1
    print("features loaded:", len(feats), "| recomputed:", recomputed)
    return feats

features = get_features(manifest)
X = np.stack([features[s] for s in manifest["scan_id"]]).astype(np.float32)
y = manifest["proxy_label"].astype(int).to_numpy()
groups = manifest["child_id"].to_numpy()
assert X.shape[1] == 9600, f"expected 9600-dim features, got {X.shape[1]}"
print("feature matrix:", X.shape, "| class balance:", np.bincount(y))

## 5. Proxy classifier (age group) — stratified child-grouped repeated CV + permutation test

A **sanity check that the frozen features carry signal**, not a result. Splits are grouped by child (no child in both train and test) and stratified by age group, repeated over seeds; a child-level label-permutation test gives a p-value that accuracy beats chance. Near chance ⇒ do not interpret any explanation.


In [ ]:
# === Train the age head with STRATIFIED child-grouped, repeated CV ===
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

def cv_splits(X, y, groups, seed=0, stratify=True):
    n = min(CFG["N_FOLDS"], len(np.unique(groups)))
    if stratify:
        try:
            return list(StratifiedGroupKFold(n_splits=n, shuffle=True,
                                             random_state=seed).split(X, y, groups))
        except Exception:
            pass
    return list(GroupKFold(n_splits=n).split(X, y, groups))

def _fit_head(Xtr, ytr):
    head = nn.Sequential(nn.Flatten(), nn.Linear(Xtr.shape[1], 512), nn.ReLU(),
                         nn.Dropout(0.5), nn.Linear(512, 2)).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=CFG["HEAD_LR"])
    lossf = nn.CrossEntropyLoss()
    xb = torch.tensor(Xtr, device=DEVICE); yb = torch.tensor(ytr, device=DEVICE)
    head.train()
    for _ in range(CFG["HEAD_EPOCHS"]):
        opt.zero_grad(); lossf(head(xb), yb).backward(); opt.step()
    head.eval(); return head

def train_head(X, y, groups):
    accs, sens, spec, cms = [], [], [], np.zeros((2, 2))
    for rep in range(CFG["N_REPEATS"]):
        for tr, te in cv_splits(X, y, groups, seed=CFG["SEED"] + rep):
            head = _fit_head(X[tr], y[tr])
            with torch.no_grad():
                pred = head(torch.tensor(X[te], device=DEVICE)).argmax(1).cpu().numpy()
            cm = confusion_matrix(y[te], pred, labels=[0, 1]); cms += cm
            tn, fp, fn, tp = cm.ravel()
            accs.append((tp + tn) / max(tp + tn + fp + fn, 1))
            sens.append(tp / max(tp + fn, 1)); spec.append(tn / max(tn + fp, 1))
    final_state = _fit_head(X, y).state_dict()   # attribution model: trained on ALL data
    return (dict(acc=np.mean(accs), acc_std=np.std(accs), sens=np.mean(sens),
                 spec=np.mean(spec), n_fits=len(accs)), np.array(accs), cms, final_state)

clf_metrics, acc_dist, cm_total, head_state = train_head(X, y, groups)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in clf_metrics.items()})

def _probe_acc(Xp, yp, groups, seed=0):
    return float(np.nanmean(cross_val_score(LogisticRegression(max_iter=1000), Xp, yp,
                                             cv=cv_splits(Xp, yp, groups, seed=seed))))
_uc = np.unique(groups); _lab = np.array([y[groups == c][0] for c in _uc])
_rng = np.random.default_rng(CFG["SEED"])
try:
    real_acc = _probe_acc(X, y, groups, seed=CFG["SEED"]); perm = []
    for _ in range(CFG["N_PERM_ACC"]):
        shuf = dict(zip(_uc, _rng.permutation(_lab)))
        yp = np.array([shuf[g] for g in groups])
        try: perm.append(_probe_acc(X, yp, groups, seed=CFG["SEED"]))
        except Exception: pass
    perm = np.array(perm)
    p_acc = (1 + int(np.sum(perm >= real_acc))) / (len(perm) + 1) if len(perm) else float("nan")
    clf_metrics["probe_acc"] = round(real_acc, 3); clf_metrics["probe_perm_p"] = round(p_acc, 4)
    print(f"proxy linear-probe acc={real_acc:.3f} | permutation p={p_acc:.4f} (n_perm={len(perm)})")
except Exception as e:
    perm = np.array([]); real_acc = float("nan")
    print("proxy permutation test skipped:", repr(e))

if clf_metrics["acc"] < 0.6:
    print("WARNING: near chance — do NOT interpret explanations until preprocessing is checked.")

model = AgeSFCN(feature_extractor).to(DEVICE)
model.head.load_state_dict(head_state); model.eval()

# Save the attribution head so the model is reusable without retraining
torch.save(head_state, DIRS["results"] / "age_head_state.pt")
print("saved age head | CV acc", round(clf_metrics["acc"], 4), "| probe acc", round(real_acc, 4) if real_acc == real_acc else None)


In [ ]:
# === Plot: CV accuracy distribution, pooled confusion, permutation null ===
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(acc_dist, bins=12, color="#4C72B0", edgecolor="white")
ax[0].axvline(clf_metrics["acc"], color="k", ls="--", label=f"mean {clf_metrics['acc']:.2f}")
ax[0].axvline(0.5, color="r", ls=":", label="chance"); ax[0].set_title(
    f"CV accuracy ({clf_metrics['n_fits']} fits)"); ax[0].set_xlabel("accuracy"); ax[0].legend(fontsize=8)

cmn = cm_total / cm_total.sum(1, keepdims=True)
im = ax[1].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax[1].set_xticks([0, 1], ["Y", "O"]); ax[1].set_yticks([0, 1], ["Y", "O"])
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("true"); ax[1].set_title("Confusion (pooled, row-norm)")
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center",
                   color="white" if cmn[i, j] > 0.5 else "black")

if perm.size:
    ax[2].hist(perm, bins=15, color="#999999", edgecolor="white", label="permuted labels")
    ax[2].axvline(real_acc, color="#C44E52", ls="--", label=f"observed {real_acc:.2f}")
    ax[2].set_title(f"Permutation null (p={clf_metrics.get('probe_perm_p','nan')})")
    ax[2].set_xlabel("probe accuracy"); ax[2].legend(fontsize=8)
else:
    ax[2].axis("off")
fig.tight_layout(); fig.savefig(DIRS["results"] / "classifier.png", bbox_inches="tight"); plt.show()

## 6. Attribution methods — pre-registered M1–M8 + exploratory M9

Metrics (faithfulness, sanity, bootstrap-by-child CIs, permutation, BH-FDR) and the 3D ReX occlusion routine are defined first, then every map is computed on the explanation set.


In [ ]:
# === Evaluation metrics (faithfulness, sanity, bootstrap-by-child, permutation, FDR) ===
# Fill may be a scalar (zero / mean) or a full-volume tensor (blur). Same code path for every fill.
from scipy.stats import spearmanr
from scipy.ndimage import gaussian_filter

BLUR_SIGMA = 4.0          # voxels on the 1 mm grid; pre-registered (plan section 10, A7)
FAITH_FILLS = ["zero", "mean", "blur"]

def fill_for(name, x):
    """Fill for one scan. zero = 0; mean = scan's mean over brain voxels (x > 0); blur = Gaussian blur of the scan."""
    if name == "zero":
        return 0.0
    if name == "mean":
        b = x[0, 0]
        return float(b[b > 0].mean().item())
    if name == "blur":
        blurred = gaussian_filter(x[0, 0].cpu().numpy(), sigma=BLUR_SIGMA, mode="nearest")
        return torch.from_numpy(blurred.astype(np.float32))[None, None].to(x.device)
    raise ValueError(name)

def _fill_like(x, fill):
    return torch.full_like(x, float(fill)) if isinstance(fill, (int, float)) else fill

@torch.no_grad()
def _curve(model, x, attribution, target, fractions, fill, mode):
    model.eval()
    F_ = _fill_like(x, fill)
    order = np.argsort(-attribution.ravel()); n = order.size; shape = x.shape[2:]; pts = []
    for frac in [0.0] + list(fractions):
        k = int(round(frac * n))
        if mode == "deletion":
            v = x.clone()
            if k:
                idx = np.unravel_index(order[:k], shape); v[0, 0][idx] = F_[0, 0][idx]
        else:
            v = F_.clone()
            if k:
                idx = np.unravel_index(order[:k], shape); v[0, 0][idx] = x[0, 0][idx]
        pts.append((frac, torch.softmax(model(v), -1)[0, target].item()))
    auc = sum((f1 - f0) * (p0 + p1) / 2 for (f0, p0), (f1, p1) in zip(pts[:-1], pts[1:]))
    return auc / (pts[-1][0] - pts[0][0]), pts

def deletion_auc(model, x, attr, target, fractions=(0.05, 0.1, 0.2, 0.3), fill=0.0):
    return _curve(model, x, attr, target, fractions, fill, "deletion")
def insertion_auc(model, x, attr, target, fractions=(0.05, 0.1, 0.2, 0.3), fill=0.0):
    return _curve(model, x, attr, target, fractions, fill, "insertion")

def randomize_model(model, seed=0):
    rnd = copy.deepcopy(model); torch.manual_seed(seed)
    for m in rnd.modules():
        if hasattr(m, "reset_parameters"):
            m.reset_parameters()
    return rnd

def spearman_maps(a, b):
    r, _ = spearmanr(a.ravel(), b.ravel()); return float(r)

def bootstrap_ci_by_child(values, child_ids, n_boot=None, alpha=0.05, seed=0):
    n_boot = n_boot or CFG["N_BOOT"]
    values = np.asarray(values, float); child_ids = np.asarray(child_ids)
    children = np.unique(child_ids); grp = {c: values[child_ids == c] for c in children}
    rng = np.random.default_rng(seed); means = np.empty(n_boot)
    for i in range(n_boot):
        pick = rng.choice(children, children.size, replace=True)
        means[i] = np.concatenate([grp[c] for c in pick]).mean()
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(values.mean()), float(lo), float(hi)

def benjamini_hochberg(pvals):
    p = np.asarray(pvals, float); m = p.size; order = np.argsort(p)
    ranked = np.minimum.accumulate((p[order] * m / (np.arange(m) + 1))[::-1])[::-1]
    adj = np.empty(m); adj[order] = np.clip(ranked, 0, 1); return adj


In [ ]:
# === 3D ReX: causal occlusion responsibility (exploratory M9) ===
@torch.no_grad()
def rex_responsibility(model, x, target, grid=(5, 6, 5), fill=0.0, batch_size=8):
    model.eval(); _, _, D, H, W = x.shape
    def prob(v): return torch.softmax(model(v), -1)[:, target]
    p0 = prob(x).item()
    def edges(size, g): return np.linspace(0, size, g + 1).round().astype(int)
    eD, eH, eW = edges(D, grid[0]), edges(H, grid[1]), edges(W, grid[2])
    boxes = [(slice(eD[i], eD[i+1]), slice(eH[j], eH[j+1]), slice(eW[k], eW[k+1]))
             for i in range(grid[0]) for j in range(grid[1]) for k in range(grid[2])]
    drops = np.zeros(len(boxes), np.float32)
    for s in range(0, len(boxes), batch_size):
        chunk = boxes[s:s + batch_size]
        occ = x.repeat(len(chunk), 1, 1, 1, 1)
        for b, (sd, sh, sw) in enumerate(chunk):
            occ[b, :, sd, sh, sw] = fill
        drops[s:s + len(chunk)] = p0 - prob(occ).cpu().numpy()
    coarse = torch.from_numpy(drops).reshape(*grid)[None, None].float()
    return F.interpolate(coarse, size=(D, H, W), mode="trilinear", align_corners=False)[0, 0].numpy()

In [ ]:
# === Compute all maps on the explanation set ===
from captum.attr import (Saliency, IntegratedGradients, NoiseTunnel,
                         LayerGradCam, LayerAttribution, GuidedBackprop)

def _vol(attr):
    return attr.detach().abs().sum(1)[0].cpu().numpy().astype(np.float32)

def lrp_zennit(model, x, target):
    from zennit.attribution import Gradient
    from zennit.composites import EpsilonPlusFlat
    with Gradient(model=model, composite=EpsilonPlusFlat()) as attr:
        out = model(x); seed = torch.zeros_like(out); seed[0, target] = 1.0
        _, rel = attr(x, seed)
    return _vol(rel)

def _free():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_maps(model, x, target):
    final_conv = model.feature_extractor.conv_5   # layer Das et al. used
    early_conv = model.feature_extractor.conv_2
    m = {}
    def gc(layer):
        a = LayerGradCam(model, layer).attribute(x, target=target)
        return _vol(LayerAttribution.interpolate(a, x.shape[2:]))
    m["M1_gradcam_final"] = gc(final_conv); _free()
    m["M2_gradcam_early"] = gc(early_conv); _free()
    m["M3_saliency"] = _vol(Saliency(model).attribute(x, target=target, abs=False)); _free()
    m["M4_integrated_gradients"] = _vol(IntegratedGradients(model).attribute(
        x, baselines=torch.zeros_like(x), target=target,
        n_steps=CFG["IG_STEPS"], internal_batch_size=1)); _free()
    m["M5_smoothgrad"] = _vol(NoiseTunnel(Saliency(model)).attribute(
        x, nt_type="smoothgrad", nt_samples=CFG["SG_SAMPLES"], nt_samples_batch_size=1,
        stdevs=CFG["SG_STDEV"], target=target)); _free()
    m["M6_vargrad"] = _vol(NoiseTunnel(Saliency(model)).attribute(
        x, nt_type="vargrad", nt_samples=CFG["SG_SAMPLES"], nt_samples_batch_size=1,
        stdevs=CFG["SG_STDEV"], target=target)); _free()
    try:
        m["M7_lrp"] = lrp_zennit(model, x, target)
    except Exception as e:
        print("M7 LRP failed, dropped:", repr(e))
    _free()
    m["M8_guided_backprop"] = _vol(GuidedBackprop(model).attribute(x, target=target)); _free()
    m["M9_rex_exploratory"] = rex_responsibility(model, x, target, grid=CFG["REX_GRID"]); _free()
    return m

# Explanation set: all children (default) or one grouped test fold.
if CFG["EXPLAIN_SET"] == "all":
    test_rows = manifest.reset_index(drop=True)
else:
    idx = next(GroupKFold(n_splits=min(CFG["N_FOLDS"], len(np.unique(groups))))
               .split(X, y, groups))[1]
    test_rows = manifest.iloc[idx].reset_index(drop=True)
print(f"explaining {len(test_rows)} scans with 9 methods (EXPLAIN_SET={CFG['EXPLAIN_SET']})")

all_maps = {}
t0 = time.time()
for i, (_, row) in enumerate(test_rows.iterrows(), 1):
    x = load_volume(row["scan_id"]).to(DEVICE)
    maps = compute_maps(model, x, int(row["proxy_label"]))
    all_maps[row["scan_id"]] = maps
    for name, arr in maps.items():
        np.save(DIRS["explanations"] / f"{row['scan_id']}__{name}.npy", arr)
    if i % 10 == 0 or i == len(test_rows):
        print(f"  {i}/{len(test_rows)} scans | {time.time()-t0:.0f}s elapsed")
print("methods per scan:", list(next(iter(all_maps.values())).keys()))

In [ ]:
# === Per-method example panel (one representative scan) ===
def mid_overlay(ax, sid, arr):
    v = np.load(DIRS["preproc"] / f"{sid}.npy")
    z = v.shape[0] // 2
    ax.imshow(v[z].T, origin="lower", cmap="gray")
    a = np.abs(arr[z]).T
    ax.imshow(np.ma.masked_less(a, np.percentile(a, 90)), origin="lower", cmap="hot", alpha=0.6)
    ax.axis("off")

ex_sid = test_rows.iloc[0]["scan_id"]
maps0 = all_maps[ex_sid]
fig, axes = plt.subplots(1, len(maps0), figsize=(2.2 * len(maps0), 2.6))
for ax, (name, arr) in zip(np.atleast_1d(axes), maps0.items()):
    mid_overlay(ax, ex_sid, arr); ax.set_title(name.split("_", 1)[0], fontsize=9)
fig.suptitle(f"All methods on one scan ({ex_sid}) — top-10% voxels over the mid-axial slice", y=1.04)
fig.savefig(DIRS["results"] / "methods_example.png", dpi=100, bbox_inches="tight"); plt.show()

In [ ]:
# === Contact sheet: the paper's method (M1 Grad-CAM) across ALL explained scans ===
def _m1_slice(sid):
    arr = all_maps[sid]["M1_gradcam_final"]
    return arr[arr.shape[0] // 2].T
fig = montage(list(all_maps.keys()), _m1_slice,
              f"M1 Grad-CAM (paper's method) — mid-axial slice, all {len(all_maps)} scans",
              cmap="hot")
fig.savefig(DIRS["results"] / "m1_contact_sheet.png", dpi=90, bbox_inches="tight"); plt.show()

## 8. Faithfulness — deletion / insertion AUC

Remove (or insert) the top-attributed voxels and track the model's age-group probability. A faithful map drops the probability fast under deletion (low deletion-AUC) and raises it fast under insertion (high insertion-AUC). 95% CIs bootstrapped **by child**.


In [ ]:
# === Deletion / insertion faithfulness per method, per fill ===
faith_rows, fill_log = [], []
for _, row in test_rows.iterrows():
    x = load_volume(row["scan_id"]).to(DEVICE); t = int(row["proxy_label"])
    for fname in FAITH_FILLS:
        fv = fill_for(fname, x)
        if fname == "mean":
            fill_log.append(dict(scan=row["scan_id"], mean_fill=fv))
        for name, arr in all_maps[row["scan_id"]].items():
            d, _ = deletion_auc(model, x, arr, t, fill=fv)
            ins, _ = insertion_auc(model, x, arr, t, fill=fv)
            faith_rows.append(dict(fill=fname, scan=row["scan_id"], child=row["child_id"],
                                   method=name, deletion_auc=d, insertion_auc=ins))
faith = pd.DataFrame(faith_rows)
faith.to_csv(DIRS["results"] / "faithfulness_per_scan_fills.csv", index=False)
pd.DataFrame(fill_log).to_csv(DIRS["results"] / "mean_fill_values.csv", index=False)
print("rows:", len(faith), "| fills:", faith.fill.unique().tolist(), "| methods:", faith.method.nunique())


In [ ]:
# === Random-order deletion/insertion baseline, per fill (one noise draw per scan, shared across fills) ===
_rng_b = np.random.default_rng(CFG["SEED"]); rb_rows = []
for _, row in test_rows.iterrows():
    x = load_volume(row["scan_id"]).to(DEVICE); t = int(row["proxy_label"])
    noise = _rng_b.standard_normal(size=tuple(x.shape[2:])).astype(np.float32)
    for fname in FAITH_FILLS:
        fv = fill_for(fname, x)
        d, _ = deletion_auc(model, x, noise, t, fill=fv)
        ins, _ = insertion_auc(model, x, noise, t, fill=fv)
        rb_rows.append(dict(fill=fname, scan=row["scan_id"], child=row["child_id"],
                            rand_deletion_auc=d, rand_insertion_auc=ins))
rb = pd.DataFrame(rb_rows)
rb.to_csv(DIRS["results"] / "random_order_baseline_fills.csv", index=False)
for fname in FAITH_FILLS:
    s = rb[rb.fill == fname]
    print(f"RANDOM-ORDER [{fname}] | deletion={s.rand_deletion_auc.mean():.3f} (sd {s.rand_deletion_auc.std():.3f})"
          f" | insertion={s.rand_insertion_auc.mean():.3f} (sd {s.rand_insertion_auc.std():.3f}) [n={len(s)}]")


In [ ]:
# === Per-fill CIs, random baseline CIs, paired ReX vs Guided Backprop, reproducibility check ===
rows_ci, rows_pair, rows_rand = [], [], []
for fname in FAITH_FILLS:
    f = faith[faith.fill == fname]
    for name in f.method.unique():
        sub = f[f.method == name].sort_values("child")
        dm, dlo, dhi = bootstrap_ci_by_child(sub.deletion_auc.tolist(), sub.child.tolist())
        im, ilo, ihi = bootstrap_ci_by_child(sub.insertion_auc.tolist(), sub.child.tolist())
        rows_ci.append(dict(fill=fname, method=name,
                            deletion_auc=round(dm, 3), del_ci_lo=round(dlo, 3), del_ci_hi=round(dhi, 3),
                            insertion_auc=round(im, 3), ins_ci_lo=round(ilo, 3), ins_ci_hi=round(ihi, 3)))
    r = rb[rb.fill == fname]
    rdm, rdlo, rdhi = bootstrap_ci_by_child(r.rand_deletion_auc.tolist(), r.child.tolist())
    rim, rilo, rihi = bootstrap_ci_by_child(r.rand_insertion_auc.tolist(), r.child.tolist())
    rows_rand.append(dict(fill=fname, deletion_auc=round(rdm, 3), del_ci_lo=round(rdlo, 3), del_ci_hi=round(rdhi, 3),
                          insertion_auc=round(rim, 3), ins_ci_lo=round(rilo, 3), ins_ci_hi=round(rihi, 3)))
    piv = f.pivot_table(index="child", columns="method", values="insertion_auc")
    diff = (piv["M9_rex_exploratory"] - piv["M8_guided_backprop"]).dropna()
    pm, plo, phi = bootstrap_ci_by_child(diff.tolist(), diff.index.tolist())
    _rng = np.random.default_rng(0); _ch = np.asarray(diff.index.tolist()); _v = np.asarray(diff.values, float)
    _g = {c: _v[_ch == c] for c in np.unique(_ch)}; _means = np.empty(CFG["N_BOOT"])
    for _i in range(CFG["N_BOOT"]):
        _pick = _rng.choice(list(_g.keys()), len(_g), replace=True)
        _means[_i] = np.concatenate([_g[c] for c in _pick]).mean()
    p_boot = 2 * min((_means <= 0).mean(), (_means >= 0).mean())
    rows_pair.append(dict(fill=fname, diff_rex_minus_gb=round(pm, 4), ci_lo=round(plo, 4), ci_hi=round(phi, 4),
                          boot_two_sided_p=round(p_boot, 4), n=len(diff)))

ci = pd.DataFrame(rows_ci); rand = pd.DataFrame(rows_rand); pair = pd.DataFrame(rows_pair)
ci.to_csv(DIRS["results"] / "faithfulness_ci_by_fill.csv", index=False)
rand.to_csv(DIRS["results"] / "random_order_ci_by_fill.csv", index=False)
pair.to_csv(DIRS["results"] / "paired_insertion_rex_vs_gb_by_fill.csv", index=False)
for fname in FAITH_FILLS:
    print(f"\n=== fill: {fname} ===")
    print(ci[ci.fill == fname].drop(columns="fill").sort_values("deletion_auc").to_string(index=False))
print("\nRANDOM-ORDER with CI:"); print(rand.to_string(index=False))
print("\nPAIRED ReX - GB insertion:"); print(pair.to_string(index=False))

# reproducibility check: zero fill must match the perscan run (deletion means, 3 dp)
ref = {"M9_rex_exploratory": 0.141, "M4_integrated_gradients": 0.528, "M8_guided_backprop": 0.556,
       "M5_smoothgrad": 0.556, "M3_saliency": 0.568, "M6_vargrad": 0.580, "M1_gradcam_final": 0.644,
       "M7_lrp": 0.674, "M2_gradcam_early": 0.777}
z = ci[ci.fill == "zero"].set_index("method").deletion_auc
diffs = {k: round(float(z[k]) - v, 3) for k, v in ref.items() if k in z.index}
print("\nZERO-FILL REPRODUCIBILITY (deletion mean minus perscan run):", diffs)
